In [2]:
from ultralytics import YOLO

# LOAD PRETRAINED YOLOv8 NANO MODEL (FP32)
model = YOLO("yolov8n.pt")
print("FP32 model loaded sucessfully")

FP32 model loaded sucessfully


In [3]:
# Validate onCOCO128 dataset (small subset of COCO)

metrics = model.val(data = "coco128.yaml")
# model.val(): runs validation
# data = "coco128.yaml": tells YOLO to:
# 1. Download COCO128 dataset automatically
# 2. Use official annotations
# 3. COmpute officail detection metrics

Ultralytics 8.4.14  Python-3.10.19 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 395.4103.5 MB/s, size: 48.7 KB)
val: Scanning C:\Users\faiza\datasets\coco128\labels\train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.2s/it 17.6s2.6s
                   all        128        929      0.639      0.536      0.607      0.448
                person         61        254      0.793      0.677      0.764      0.538
               bicycle          3          6      0.514      0.333      0.315      0.264
                   car         12         46      0.813      0.217      0.272      0.167
            motorcycle          4          5      0.687      0.887      0.898      0.718
              airplane 

In [4]:
print("mAP50:", metrics.box.map50)
# Mean Average Presicion at IoU = 0.5
# Basic detection quality measure.

print("mAP50-95:", metrics.box.map)
# Stricter metric (official COCO metric)
# More realistic performance

print("Precision:", metrics.box.mp)
# Of predicted boxes, how many are correct?

print("Recall:", metrics.box.mr)
# Of actual objects, how many were detected?

mAP50: 0.6071667846709387
mAP50-95: 0.4477848806493725
Precision: 0.6385015839516784
Recall: 0.5360858270340833


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
#  FP32 Pytorch Model

# Export to ONNX format
model.export(format="onnx") #
# Converted the PyTorch FP32 model to ONNX FP32 model

Ultralytics 8.4.14  Python-3.10.19 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.20.1 opset 22...


C:\Users\faiza\miniconda3\envs\edgeai\lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success  3.1s, saved as 'yolov8n.onnx' (12.3 MB)

Export complete (3.5s)
Results saved to C:\Users\faiza\Documents\edge-perception-project
Predict:         yolo predict task=detect model=yolov8n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolov8n.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


'yolov8n.onnx'

##### yolov8.pt -> FP32 PyTorch Model
##### yolov8n.onnx -> FP32 ONNX model
##### yolov8n_int8.onnx ->INT8 Quantized Model

This process is called Post-Training Static Quantization

In [6]:
# Create Quantization Cell

from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType

# This imports specific tools from ONNX Runtime's quantization module.

# quantize_static -> function that performs INT8 conversation
# takes the model
# uses calibration data
# produces as INT8 model

# Performs weight quantization, activation quantization, scale+ zero-point calculation

# CalibrationDataReader -> feeds calibration images
# This is a base class.
# We must create our own class that inherits from it.
# Why? Because ONNX Runtime needs a way to:
# Ask: "Give me next calibration sample.", So we implement a custom reader that feeds images.

# QuantType -> defines INT8 type

# Defines numeric type used in Quantization.
# For example:

# QuantType.QInt8 -> signed 8-bit integers
# QuantType.QUInt8 -> insigned 8-bit integers

# we chose QInt8. Why? Because signed int8 is standard for weight quantization.

# Quantization requires calibration data
# This means we must pass sample images to determine scaling ranges.

import onnx
# Used to manipulate ONNX models.

import os
# Used for file system operations (not heavily used here)

In [7]:
# Create Calibration Data Reader

import numpy as np
import cv2
from pathlib import Path

class YOLOCalibrationDataReader(CalibrationDataReader):
    # This creates a custom class.
    # It inherits from:
    # CalibrationDataReader

    # Inheritence means: We are extending functionality of an existing class.
    
    def __init__(self, image_folder, input_name, max_images=50):
        # init runs when the clas  is created .

        # Parameters:

        # image_folder: where the calibration images are stored
        # input_name: name of the model input tensor
        # max_images: number of images to use
        
        self.image_folder = Path(image_folder)
        # stores folder path as Path object.
        # Path is cleaner than string paths.
        
        self.input_name = input_name
        # Stores model input name.
        # ONNX models expect a dictionary like:
        #  {"images": tensor}
        
        self.image_paths = list(self.image_folder.glob("*.jpg"))[:max_images]
        # glob("*.jpg"): finds all .jpg files.
        # list(....): converts to lsit.
        # [:max_images]: takes first 50 images

        # Why only 50?, Calibration does not nedd full dataset.
        #  50 to 100 images are enough to estimate activation ranges.
        
        self.data_iter = iter(self.image_paths)
        # Creates iterator
        # This allows sequintial access:

        # image1 -> image2 -> image3

    def get_next(self):
        # This is required by ONNX RUntime.
        # ONNX Runtime repeatedly calls get_next() to fetch calibration data.
        
        try:
            image_path = next(self.data_iter)
            # Gets next image path. If no images left -> StopIteration error.
            
            img = cv2.imread(str(image_path))
            # Reads image using OpenCV.
            # Returns: Height x Width x Channels (HWC format)
            
            img = cv2.resize(img, (640, 640))
            # Resizes to model input size
            # YOLOv8 expects 640x640
            
            img = img.astype(np.float32) / 255.0
            # Two operations:
            # 1. Convert to float32
            # 2. Normalize pixel values from: 0-255 -> 0-1
            # This matches model training preprocessing
            
            img = np.transpose(img, (2, 0, 1))  #HWC to CHW
            # Changes image format.
            # Original: HWC
            # Model expects: CHW(Channels,Height,Width)

            # So(2,0,1) means: Take axis 2 -> axis 0, axis 0 -> 1, axis 1 -> 2
            # Reorder dimensions
            
            img = np.expand_dims(img, axis=0)
            # Adds batch dimension.
            # Now batch shape becomes: (1,3,640,640), 1 = batch size
            
            return {self.input_name: img}
            # Return dictionary.
            # Example: {"images": img}

            # ONNX Runtime expects input as dictionary

        # If image finish:
        except StopIteration:            
            return None
            # Returning None tells ONNX Runtime: Calibration finished. 

# What does this code does (conceptually):

# Quantization needs sample images .

# This class:
#Loads images from COCO128 folder 
# Resizes to 640x640
# Normalizes pixel values
# Converts format to model input shape
# Feeds them to ONNX runtime

# We limit to 50 images. That is enough for calibration

In [8]:
# Identify Model Input Name 

import onnxruntime as ort

onnx_model_path = "yolov8n.onnx"
session = ort.InferenceSession(onnx_model_path)
# Creates ONNX Runtime session
# Loads model into memory.

input_name = session.get_inputs()[0].name
# get_inputs() -> returns list of input tensors
# [0]: first input
# .name: tensor name

# We must pass this exact name during quantization.
# Otherwise ONNX Runtime cannot map data correctly.

print("Model input name:", input_name)

Model input name: images


In [9]:
# Perform INT8 Quantization

quantized_model_path = "yolov8n_int8.onnx"

calibration_data_reader = YOLOCalibrationDataReader(
    image_folder="C:/Users/faiza/datasets/coco128/images/train2017",
    input_name=input_name,
    max_images=50
)

# quantize_static(: This is the core function.
quantize_static(
    model_input=onnx_model_path,
    # Path to FP32 ONNX model
    
    model_output=quantized_model_path,
    # Path where INT8 model will be saved.
    
    calibration_data_reader= calibration_data_reader,
    # Instance of our class
    # This feeds calibration images.
    
    weight_type=QuantType.QInt8
    # Tells system:
    # Convert weights to signed INT8
)

print("INT8 Quantization complete.")

# What Happens internally

# During Quantization:
# 1. Model runs on Calibration
# 2. Collects activation statistics
# 3. Determines min/max ranges
# 4. Computes scaling factors
# 5. Converts FP32 weights to INT8
# 6. Stores scale + zero-point each layer

# So int8 model still computes approximate FP32 outputs using scaled integers.

# Why Calibration is Needed 
# If we directly compress values:
#  Precision would collapse.

# Calibration Allows:
# Proper scaling
# Reduced information loss
# Better accuracy retention


INT8 Quantization complete.


#### I performed:

##### Post-training static quantization

Meaning:

1. No retraining
2. No fine-tuning
3. Just compressing weights
4. Calibrating using sample images

In [10]:
from ultralytics import YOLO
int8_model = YOLO("yolov8n_int8.onnx")
metrics_int8 = int8_model.val(data="coco128.yaml")

print("INT8 mAP50:", metrics_int8.box.map50)
print("INT8 mAP50-95:", metrics_int8.box.map)
print("INT8 Precision:", metrics_int8.box.mp)
print("INT8 Recall:", metrics_int8.box.mr)

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.14  Python-3.10.19 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
Loading yolov8n_int8.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.10.0 ms, read: 251.5132.7 MB/s, size: 51.6 KB)
val: Scanning C:\Users\faiza\datasets\coco128\labels\train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 4.2it/s 30.1s0.2s
                   all        128        929          0          0          0          0
                person         61        254          0          0          0          0
               bicycle          3       

#### Manual static quantization using ONNX Runtime resulted in mAP collapse (0.0 across metrics), likely due to improper activation scaling in detection heads.

### OpenVINO INT8 Quantization


In [11]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.export(
    format="openvino",
    int8=True,
    data="coco128.yaml"
)

Ultralytics 8.4.14  Python-3.10.19 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

OpenVINO: starting export with openvino 2026.0.0-20965-c6d6a13a886-releases/2026/0...
OpenVINO: collecting INT8 calibration images from 'data=coco128.yaml'
Fast image access  (ping: 0.10.0 ms, read: 89.266.7 MB/s, size: 31.0 KB)
Scanning C:\Users\faiza\datasets\coco128\labels\train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128  0.0s
WARNING OpenVINO: >300 images recommended for INT8 calibration, found 128 images.
INFO:nncf:8 ignored nodes were found by patterns in the NNCFGraph
INFO:nncf:1 ignored nodes were found by types in the NNCFGraph
INFO:nncf:Not adding activation input quantizer for operation: 188 __module.model.22/aten::sub/Subtract
INFO:nncf:Not addin

C:\Users\faiza\miniconda3\envs\edgeai\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\faiza\miniconda3\envs\edgeai\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

OpenVINO: export success  63.6s, saved as 'yolov8n_int8_openvino_model\' (3.6 MB)

Export complete (64.1s)
Results saved to C:\Users\faiza\Documents\edge-perception-project
Predict:         yolo predict task=detect model=yolov8n_int8_openvino_model imgsz=640 int8
Validate:        yolo val task=detect model=yolov8n_int8_openvino_model imgsz=640 data=coco.yaml int8 
Visualize:       https://netron.app


'yolov8n_int8_openvino_model'

In [13]:
from ultralytics import YOLO

int8_model = YOLO("yolov8n_int8_openvino_model")

metrics_int8 = int8_model.val(data="coco128.yaml", imgsz=640)

print("INT8 mAP50:", metrics_int8.box.map50)
print("INT8 mAP50-95:", metrics_int8.box.map)
print("INT8 Precision:", metrics_int8.box.mp)
print("INT8 Recall:", metrics_int8.box.mr)

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.14  Python-3.10.19 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
Loading yolov8n_int8_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on (CPU)...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.10.0 ms, read: 322.6198.0 MB/s, size: 48.1 KB)
val: Scanning C:\Users\faiza\datasets\coco128\labels\train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 7.5it/s 17.2s0.2s
                   all        128        929      0.681      0.545      0.617      0.456
                person         61        254      0.817      0.654      0.757      0.532
               bicycle  